# 09 — Limitations & Research Constraints
**Critical documentation for the paper — required for academic integrity.**

This notebook catalogs every known limitation of the EDA for Music framework. Each section is structured for copy-paste into the paper's Limitations section.

---

In [1]:
import sys
from pathlib import Path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import json
import pandas as pd

# Load summary data for reference
ex6_df = pd.read_csv(ROOT / "data/processed/ex6_signal_report_card.csv")
with open(ROOT / "data/processed/gnn_training_summary.json") as f:
    gnn_summary = json.load(f)

print("Loaded exercise 6 results and GNN summary.")
print(ex6_df[["Artist", "Type", "Score", "Verdict"]].to_string(index=False))

Loaded exercise 6 results and GNN summary.
               Artist    Type  Score        Verdict
 Relaxing White Noise   GHOST 0.3566 LIKELY_ORGANIC
Meditation Relax Club   GHOST 0.3313 LIKELY_ORGANIC
                Calmo   GHOST 0.2201 LIKELY_ORGANIC
           Nils Frahm ORGANIC 0.0233 LIKELY_ORGANIC


## L1 — Dataset Scale (Critical)

In [2]:
limitations = {}

limitations["L1_dataset_scale"] = {
    "title": "Dataset Scale — Critically Small",
    "severity": "HIGH",
    "description": (
        "The Neo4j graph contains only 3 confirmed ghost artist candidates and 0 confirmed organic "
        "artists (Nils Frahm, the organic control, is not in the database). "
        "The GNN is trained on 65 nodes total: 14 ghost labels (3 real + 11 Kaggle candidates) "
        "and 51 organic labels (from Kaggle audio feature analysis). "
        "The Kaggle ghost candidates are identified by near-zero audio feature variance — "
        "a necessary but not sufficient condition for being a ghost artist."
    ),
    "impact": (
        "The 100% test accuracy (AUC-ROC=1.0) is almost certainly optimistic. "
        "With 65 nodes and a 60/20/20 split, the test set has ~13 nodes. "
        "The model may be memorising graph structure rather than learning generalizable patterns. "
        "Reported as 'proof of concept' — not a production classifier."
    ),
    "mitigation": (
        "Results are framed as 'proof of concept' in the paper. "
        "All performance metrics are reported with the caveat that n=65 is too small "
        "for reliable generalization estimates. "
        "Leave-one-out cross-validation on the 4 real artists confirms the rank ordering "
        "(ghost > organic) even without the Kaggle augmentation."
    ),
    "paper_language": (
        "Our GNN evaluation is constrained by the available ground-truth dataset of n=3 confirmed "
        "ghost artist candidates from journalistic investigations. We augment with Kaggle-derived "
        "pseudo-labels, but caution that reported metrics (Acc=1.0, AUC=1.0) reflect "
        "near-separable training data, not deployment-grade accuracy."
    ),
}

print(f"L1: {limitations['L1_dataset_scale']['title']}")
print(f"Severity: {limitations['L1_dataset_scale']['severity']}")
print(f"\nImpact: {limitations['L1_dataset_scale']['impact']}")
print(f"\nPaper language:\n  {limitations['L1_dataset_scale']['paper_language']}")

L1: Dataset Scale — Critically Small
Severity: HIGH

Impact: The 100% test accuracy (AUC-ROC=1.0) is almost certainly optimistic. With 65 nodes and a 60/20/20 split, the test set has ~13 nodes. The model may be memorising graph structure rather than learning generalizable patterns. Reported as 'proof of concept' — not a production classifier.

Paper language:
  Our GNN evaluation is constrained by the available ground-truth dataset of n=3 confirmed ghost artist candidates from journalistic investigations. We augment with Kaggle-derived pseudo-labels, but caution that reported metrics (Acc=1.0, AUC=1.0) reflect near-separable training data, not deployment-grade accuracy.


## L2 — Label Quality

In [3]:
limitations["L2_label_quality"] = {
    "title": "Label Quality — Journalist-Derived, Not Platform-Confirmed",
    "severity": "MEDIUM",
    "description": (
        "Ghost labels for Relaxing White Noise and Meditation Relax Club come from "
        "the Michael Smith wire fraud case (2024, DOJ indictment) and the Dagens Nyheter "
        "investigation (2023). Calmo is labeled ghost based on signal scores alone. "
        "None of these labels are confirmed by Spotify's internal fraud systems."
    ),
    "impact": (
        "Some 'ghost' artists may be legitimate niche producers (e.g., sleep-focused ambient music "
        "with high release cadence) rather than fraudulent stream farmers. "
        "Calmo in particular shows moderate signal scores and has cross-platform YouTube presence, "
        "making it ambiguous."
    ),
    "mitigation": (
        "We use the term 'ghost candidate' rather than 'confirmed ghost' in the paper. "
        "The framework is validated against the DOJ/Dagens Nyheter cases as positive examples "
        "and Nils Frahm (established commercial artist, multiple label releases) as a negative example."
    ),
    "paper_language": (
        "Ghost labels in this study derive from public journalistic and legal records rather than "
        "platform-internal fraud flags. We adopt the conservative framing of 'ghost candidate' "
        "to reflect this uncertainty, and acknowledge that high-cadence ambient production "
        "may overlap with legitimate micro-genre activity."
    ),
}

print(f"L2: {limitations['L2_label_quality']['title']}")
print(f"\nPaper language:\n  {limitations['L2_label_quality']['paper_language']}")

L2: Label Quality — Journalist-Derived, Not Platform-Confirmed

Paper language:
  Ghost labels in this study derive from public journalistic and legal records rather than platform-internal fraud flags. We adopt the conservative framing of 'ghost candidate' to reflect this uncertainty, and acknowledge that high-cadence ambient production may overlap with legitimate micro-genre activity.


## L3 — Signal Gaps (API Restrictions)

In [4]:
limitations["L3_signal_gaps"] = {
    "title": "Signal Gaps from February 2026 API Lockdown",
    "severity": "HIGH",
    "description": (
        "Spotify's February 2026 API changes removed or restricted several key endpoints:\n"
        "  - audio-features: Returns HTTP 403 for all non-premium developer accounts\n"
        "  - related-artists: Returns HTTP 403 \n"
        "  - followers: Stripped from all artist objects\n"
        "  - /playlists/{id}/tracks: Returns 404 for editorial playlists"
    ),
    "signal_impact": {
        "S1_audio_fingerprint": (
            "Adapted: Uses Kaggle 114K-track dataset instead of live API. "
            "Hit rate = 0% for all 3 ghost artists (niche genre not in Kaggle). "
            "Signal returns None for all analyzed artists — effectively disabled."
        ),
        "S3_playlist_cooccurrence": (
            "Adapted: Uses ISRC prefix Jaccard similarity as proxy (Neo4j-based). "
            "Original signal (shared playlist appearance) is completely inaccessible. "
            "The proxy measures production company overlap, not playlist co-occurrence."
        ),
        "S4_follower_ratio": (
            "Adapted: Renamed 'Catalog Density Anomaly' — follower data unavailable. "
            "Uses tracks_per_day upload velocity as primary signal instead of follower:catalog ratio."
        ),
        "S6_graph_density": (
            "Adapted: Uses ISRC HHI (Herfindahl-Hirschman Index) instead of related-artist graph. "
            "Actually a stronger signal because ISRC records are financial/legal, not algorithmic."
        ),
    },
    "paper_language": (
        "The February 2026 Spotify API update restricted access to audio features, "
        "follower counts, related artists, and editorial playlist tracks. "
        "We adapted Signals 1, 3, 4, and 6 to use cached pre-2026 data and ISRC-based proxies. "
        "Notably, the ISRC-HHI proxy for Signal 6 proved more discriminative than the "
        "originally planned related-artist graph centrality measure."
    ),
}

print("L3: API Signal Gaps")
for sig, impact in limitations["L3_signal_gaps"]["signal_impact"].items():
    print(f"\n  {sig}:")
    print(f"    {impact}")
print(f"\nPaper language:\n  {limitations['L3_signal_gaps']['paper_language']}")

L3: API Signal Gaps

  S1_audio_fingerprint:
    Adapted: Uses Kaggle 114K-track dataset instead of live API. Hit rate = 0% for all 3 ghost artists (niche genre not in Kaggle). Signal returns None for all analyzed artists — effectively disabled.

  S3_playlist_cooccurrence:
    Adapted: Uses ISRC prefix Jaccard similarity as proxy (Neo4j-based). Original signal (shared playlist appearance) is completely inaccessible. The proxy measures production company overlap, not playlist co-occurrence.

  S4_follower_ratio:
    Adapted: Renamed 'Catalog Density Anomaly' — follower data unavailable. Uses tracks_per_day upload velocity as primary signal instead of follower:catalog ratio.

  S6_graph_density:
    Adapted: Uses ISRC HHI (Herfindahl-Hirschman Index) instead of related-artist graph. Actually a stronger signal because ISRC records are financial/legal, not algorithmic.

Paper language:
  The February 2026 Spotify API update restricted access to audio features, follower counts, related art

## L4 — Cross-Platform Assumption Failure (S7)

In [5]:
limitations["L4_cross_platform"] = {
    "title": "Cross-Platform Invisibility Assumption Rejected",
    "severity": "MEDIUM",
    "description": (
        "Signal 7 was designed under the assumption that ghost artists lack cross-platform presence. "
        "Testing revealed this assumption is false for the ambient/relaxation sub-genre:\n"
        "  - Relaxing White Noise: 353,775,028 YouTube views\n"
        "  - Meditation Relax Club: 157,581,269 YouTube views\n"
        "  - Nils Frahm (organic control): 9,107,596 YouTube views\n"
        "  - Calmo (ghost candidate): 155 YouTube views"
    ),
    "impact": (
        "S7 correctly identifies Calmo as suspicious (low cross-platform presence) "
        "but incorrectly rates RWN and MRC as organic (high YouTube presence). "
        "The signal is anti-correlated with ghost status for these two artists. "
        "This reduces S7 weight to 0.10 in the final verdict scorer."
    ),
    "revised_hypothesis": (
        "Ghost behavior in the relaxation genre is Spotify-stream-farming, not platform isolation. "
        "These artists built genuine YouTube audiences and then began fraudulently inflating "
        "Spotify stream counts. The cross-platform signal is genre-dependent."
    ),
    "paper_language": (
        "Our initial hypothesis that ghost artists exhibit cross-platform invisibility was "
        "refuted by Signal 7 results: the two primary ghost candidates (RWN, MRC) each exceed "
        "100M YouTube views. We revise our understanding: ghost behavior in the ambient micro-genre "
        "reflects Spotify-specific economic incentivization rather than fabricated artist identities. "
        "Signal 7 retains discriminative power for genres without established YouTube economies "
        "(e.g., AI-generated lo-fi, ambient drone) but is unreliable for this sub-genre."
    ),
}

print("L4: Cross-Platform Assumption Failure")
print(f"\nRevised hypothesis: {limitations['L4_cross_platform']['revised_hypothesis']}")
print(f"\nPaper language:\n  {limitations['L4_cross_platform']['paper_language']}")

L4: Cross-Platform Assumption Failure

Revised hypothesis: Ghost behavior in the relaxation genre is Spotify-stream-farming, not platform isolation. These artists built genuine YouTube audiences and then began fraudulently inflating Spotify stream counts. The cross-platform signal is genre-dependent.

Paper language:
  Our initial hypothesis that ghost artists exhibit cross-platform invisibility was refuted by Signal 7 results: the two primary ghost candidates (RWN, MRC) each exceed 100M YouTube views. We revise our understanding: ghost behavior in the ambient micro-genre reflects Spotify-specific economic incentivization rather than fabricated artist identities. Signal 7 retains discriminative power for genres without established YouTube economies (e.g., AI-generated lo-fi, ambient drone) but is unreliable for this sub-genre.


## L5 — Generalizability Across Genres

In [6]:
limitations["L5_generalizability"] = {
    "title": "Genre Scope — Ambient/Relaxation Only",
    "severity": "MEDIUM",
    "description": (
        "All three seed ghost artists operate in the ambient/relaxation/sleep music micro-genre. "
        "The framework has not been tested on other ghost-prone genres:\n"
        "  - Lo-fi hip hop (high volume, repetitive)\n"
        "  - AI-generated jazz/classical\n"
        "  - Fake pop artists (completely fabricated identities)\n"
        "  - Regional language streams (Turkish, Hindi)"
    ),
    "impact": (
        "Signal calibration (thresholds, weights) may be genre-specific. "
        "For example, the tracks_per_day threshold of 0.15 that flags RWN as suspicious "
        "may be normal for prolific lo-fi producers. "
        "Similarly, S5 ghost keyword detection uses terms like 'relaxing', 'sleep', 'calm' "
        "that are specific to this genre."
    ),
    "paper_language": (
        "This study's empirical results are grounded in the ambient/relaxation micro-genre "
        "due to the availability of confirmed cases from public legal proceedings. "
        "We expect the structural signals (ISRC HHI, release cadence synchrony) to generalize "
        "across genres, while content-specific signals (S5 metadata keywords, S7 YouTube presence) "
        "require genre-specific recalibration. Cross-genre validation is left for future work."
    ),
}

print("L5: Generalizability")
print(f"\nPaper language:\n  {limitations['L5_generalizability']['paper_language']}")

L5: Generalizability

Paper language:
  This study's empirical results are grounded in the ambient/relaxation micro-genre due to the availability of confirmed cases from public legal proceedings. We expect the structural signals (ISRC HHI, release cadence synchrony) to generalize across genres, while content-specific signals (S5 metadata keywords, S7 YouTube presence) require genre-specific recalibration. Cross-genre validation is left for future work.


## L6 — GNN Architecture Constraints

In [7]:
limitations["L6_gnn_constraints"] = {
    "title": "GNN Training Data and Architecture Limitations",
    "severity": "HIGH",
    "description": (
        "The GNN dataset uses simulated graph structure:\n"
        "  - Ghost-ghost edges: fully connected (simulated coordination assumption)\n"
        "  - Organic-organic edges: k-NN by feature similarity (k=5)\n"
        "  - No cross-class edges (clean separation assumption)\n"
        "The real graph (Neo4j) has SIMILAR_TO edges = 0, SAME_HOST edges = 0 "
        "for the 3 seed artists. The graph structure is entirely synthetic."
    ),
    "impact": (
        "Feature importance permutation yields 0.0 accuracy drop for all features — "
        "the model's 100% accuracy derives from graph structure (ghost nodes are fully connected, "
        "organic nodes are in smaller k-NN clusters), not from the node feature values. "
        "This means the model has learned: 'if you're in the large clique, you're a ghost.' "
        "This is circular: the graph structure was constructed from labels."
    ),
    "honest_assessment": (
        "The GNN results are not a valid empirical contribution as currently constructed. "
        "The synthetic graph structure makes the classification trivial. "
        "The paper frames this as: (1) a proof-of-concept pipeline, and (2) a demonstration "
        "that once a real artist co-occurrence graph is available (e.g., from shared playlist data), "
        "the GAT framework can be applied directly."
    ),
    "paper_language": (
        "Our GNN evaluation uses a graph with simulated edge structure due to the "
        "unavailability of a real artist co-occurrence graph (Spotify's related-artist "
        "and playlist-track endpoints were restricted in February 2026). "
        "We fully acknowledge that the reported 100% test accuracy reflects the "
        "constructed graph topology rather than learned feature representations — "
        "feature permutation importance is uniformly zero. "
        "This work demonstrates the architectural pipeline for when real co-occurrence "
        "data becomes available, and serves as a baseline for future graph-based approaches."
    ),
}

print("L6: GNN Architecture Constraints")
print(f"\nHonest assessment: {limitations['L6_gnn_constraints']['honest_assessment']}")
print(f"\nPaper language:\n  {limitations['L6_gnn_constraints']['paper_language']}")

L6: GNN Architecture Constraints

Honest assessment: The GNN results are not a valid empirical contribution as currently constructed. The synthetic graph structure makes the classification trivial. The paper frames this as: (1) a proof-of-concept pipeline, and (2) a demonstration that once a real artist co-occurrence graph is available (e.g., from shared playlist data), the GAT framework can be applied directly.

Paper language:
  Our GNN evaluation uses a graph with simulated edge structure due to the unavailability of a real artist co-occurrence graph (Spotify's related-artist and playlist-track endpoints were restricted in February 2026). We fully acknowledge that the reported 100% test accuracy reflects the constructed graph topology rather than learned feature representations — feature permutation importance is uniformly zero. This work demonstrates the architectural pipeline for when real co-occurrence data becomes available, and serves as a baseline for future graph-based approa

## Summary: All Limitations

In [8]:
print("=" * 80)
print("LIMITATIONS SUMMARY FOR PAPER")
print("EDA for Music — INFO 7390 — Spring 2026")
print("=" * 80)

severity_order = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
sorted_lims = sorted(limitations.items(), key=lambda x: severity_order.get(x[1]["severity"], 3))

for key, lim in sorted_lims:
    print(f"\n[{lim['severity']}] {lim['title']}")
    print("-" * 60)
    print(f"Paper copy:\n{lim['paper_language']}")

print("\n" + "=" * 80)
print("RECOMMENDED PAPER STRUCTURE FOR LIMITATIONS SECTION:")
print("=" * 80)
print("""
6. Limitations

6.1 Dataset Scale and Label Quality
  - n=3 confirmed ghost artists; GNN test accuracy must be interpreted as proof-of-concept
  - Labels derived from public journalism/legal records, not platform ground truth

6.2 API Availability Constraints  
  - Feb 2026 API lockdown removed audio features, followers, related-artists endpoints
  - Signals 1, 3, 4, 6 adapted to ISRC-based proxies; Signal 1 effectively disabled

6.3 Cross-Platform Signal Validity
  - S7 assumption (ghost = invisible cross-platform) rejected for ambient sub-genre
  - RWN=353M, MRC=157M YouTube views; signal weight reduced to 0.10

6.4 Genre Generalizability
  - Results grounded in ambient/relaxation; genre-specific recalibration needed for other genres

6.5 GNN Graph Structure
  - Synthetic edge construction (real graph is unavailable) makes accuracy metrics circular
  - Feature importance uniformly zero; architecture validated as pipeline demonstration only
""")

LIMITATIONS SUMMARY FOR PAPER
EDA for Music — INFO 7390 — Spring 2026

[HIGH] Dataset Scale — Critically Small
------------------------------------------------------------
Paper copy:
Our GNN evaluation is constrained by the available ground-truth dataset of n=3 confirmed ghost artist candidates from journalistic investigations. We augment with Kaggle-derived pseudo-labels, but caution that reported metrics (Acc=1.0, AUC=1.0) reflect near-separable training data, not deployment-grade accuracy.

[HIGH] Signal Gaps from February 2026 API Lockdown
------------------------------------------------------------
Paper copy:
The February 2026 Spotify API update restricted access to audio features, follower counts, related artists, and editorial playlist tracks. We adapted Signals 1, 3, 4, and 6 to use cached pre-2026 data and ISRC-based proxies. Notably, the ISRC-HHI proxy for Signal 6 proved more discriminative than the originally planned related-artist graph centrality measure.

[HIGH] GNN Tr